# DSC0152 - notebook v61

Notebook ini mereproduksi seluruh kandidat `submission_v61_*.csv` secara utuh.

## Metrik

Kompetisi dinilai dengan KS. Ambang KS berada di sekitar p = 0.139, sama dengan base rate 0.136, jadi hanya urutan peringkat di bagian atas yang memengaruhi skor.

## Rantai yang dijalankan

1. `features.build()` dari `features.py` (tidak diubah) -> `agas/shared/cache/features_cache.pkl`.
2. `agas/shared/feat_extra.py` -> blok fitur baru `agas/shared/cache/extra_cache.pkl`.
3. `agas/v61/training/train_submission_v61.py` -> 10 fold x 3 fold seed x 3 seed LightGBM, CatBoost, XGBoost, spline linear -> `agas/v61/output/preds_v61.npz`.
4. `agas/v61/training/make_submission_v61_final.py` -> rank blend dan file submission.

## Aturan

Hanya data kompetisi lokal, preprocessing dan feature engineering manual, tanpa AutoML atau AutoFE. Factor loading dihitung per fold dari label fold training saja, jadi tidak ada leakage. Tidak ada data eksternal dan tidak ada push ke GitHub.

`RUN_TRAINING = False` membuat notebook hanya membaca cache yang sudah ada sehingga bisa diverifikasi tanpa menghitung ulang.

In [1]:
import os, sys, json, time
from pathlib import Path

import numpy as np
import pandas as pd

RUN_TRAINING = False
FOLDS = 10
FOLD_SEEDS = '42,7,13'
LGB_SEEDS = '0,1,2'
LR = 0.01
L2 = 5.0

ROOT = Path.cwd()
AGAS = ROOT / 'agas'
VER = AGAS / 'v61'
OUT = VER / 'output'
SHARED = AGAS / 'shared'
CACHE = SHARED / 'cache'
TRAINING = VER / 'training'
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(SHARED))
os.chdir(ROOT)
print('numpy', np.__version__, '| pandas', pd.__version__)
print('version dir', VER)

numpy 2.5.1 | pandas 3.0.5
version dir /Users/agas/Documents/ITB/Lomba/dsc-logika-ui-2026/agas/v61


## 1. Data dan batasan

Tujuh tabel lokal dipakai: aplikasi, profil, aplikasi sebelumnya, riwayat cicilan, perilaku rekening, observasi biro kredit, dan transaksi e-commerce. Hanya `data/` yang dibaca.

In [2]:
import lightgbm, xgboost, catboost, sklearn

print('lightgbm', lightgbm.__version__)
print('xgboost', xgboost.__version__)
print('catboost', catboost.__version__)
print('sklearn', sklearn.__version__)

for name in ['train_applications', 'test_applications', 'customer_profiles',
             'previous_applications', 'repayment_history', 'account_behavior',
             'bureau_observations', 'ecommerce_transactions']:
    p = ROOT / 'data' / f'{name}.csv'
    print(f'{name:24s} {p.stat().st_size / 1e6:9.1f} MB')

apps = pd.read_csv(ROOT / 'data/train_applications.csv')
print('train rows', len(apps), 'base rate', round(apps.target.mean(), 5))

lightgbm 4.7.0
xgboost 3.3.0
catboost 1.2.10
sklearn 1.9.0
train_applications             8.2 MB
test_applications              3.5 MB
customer_profiles              6.4 MB
previous_applications         26.7 MB
repayment_history            159.7 MB
account_behavior             100.0 MB
bureau_observations          114.7 MB
ecommerce_transactions       528.8 MB
train rows 70000 base rate 0.13581


## 2. Feature frame

`features.build()` sudah dipakai tim sebelumnya dan tidak diubah. Lanjutannya adalah blok fitur baru di `agas/shared/feat_extra.py`, semuanya diturunkan dari tabel mentah dan stempel waktu keputusan, tanpa memakai label.

In [3]:
import features as ft

cache = CACHE / 'features_cache.pkl'
if RUN_TRAINING or not cache.exists():
    f = ft.build()
    cache.parent.mkdir(parents=True, exist_ok=True)
    f.to_pickle(cache)
else:
    f = pd.read_pickle(cache)

ex = pd.read_pickle(CACHE / 'extra_cache.pkl')
add = [c for c in ex.columns if c != 'application_id' and c not in f.columns]
f = f.merge(ex[['application_id'] + add], on='application_id', how='left')

tr = f[f.target.notna()].reset_index(drop=True)
te = f[f.target.isna()].reset_index(drop=True)
print('train', tr.shape, '| test', te.shape)

prefixes = ['bx_', 'rx_', 'px_', 'ax_', 'ex_', 'ix_', 'r2_', 'b2_', 'a2_', 'e2_']
new_cols = [c for c in tr.columns if c.startswith(tuple(prefixes))]
print('blok fitur baru:', len(new_cols))
print(sorted({c.split('_')[0] for c in new_cols}))

train (70000, 557) | test (30000, 557)
blok fitur baru: 168
['a2', 'ax', 'b2', 'bx', 'e2', 'ex', 'ix', 'px', 'r2', 'rx']


## 3. Training ensemble

Setiap model dilatih pada 10 fold StratifiedKFold, diulang pada tiga fold seed, dan LightGBM diulang pada tiga seed. Factor features dihitung ulang di dalam setiap fold dengan loading yang hanya memakai label fold training.

In [4]:
cmd = (f'python3 {TRAINING / "train_submission_v61.py"} --folds {FOLDS} '
       f'--fold-seeds {FOLD_SEEDS} --lgb-seeds {LGB_SEEDS} --lr {LR} --l2 {L2} '
       f'--out-prefix v61')
print(cmd)
if RUN_TRAINING:
    os.system(cmd)
else:
    print('dilewati, memakai agas/v61/output/preds_v61.npz yang sudah ada')

python3 /Users/agas/Documents/ITB/Lomba/dsc-logika-ui-2026/agas/v61/training/train_submission_v61.py --folds 10 --fold-seeds 42,7,13 --lgb-seeds 0,1,2 --lr 0.01 --l2 5.0 --out-prefix v61
dilewati, memakai agas/v61/output/preds_v61.npz yang sudah ada


In [5]:
from scipy.stats import ks_2samp
from sklearn.metrics import roc_auc_score

z = np.load(OUT / 'preds_v61.npz', allow_pickle=True)
train_ids, test_ids = z['train_ids'], z['test_ids']
ymap = apps.set_index('application_id').reindex(pd.Index(train_ids))
y = ymap['target'].values.astype(int)

base_rows = []
for k in sorted(c for c in z.files if c.startswith('oof_')):
    name = k[4:]
    p = z[k]
    base_rows.append({'model': name,
                      'auc': roc_auc_score(y, p),
                      'ks': ks_2samp(p[y == 1], p[y == 0]).statistic})
base = pd.DataFrame(base_rows).sort_values('ks', ascending=False)
base

,model,auc,ks
6,xgb_full,0.846485,0.525240
0,cat_full,0.845928,0.525236
2,lgb_full,0.846647,0.524615
4,lgbx_full,0.846351,0.524416
3,lgb_lean,0.846566,0.523214
1,fac_full,0.846542,0.523080
5,linear,0.842667,0.522833


## 4. Blend

Arah ekstrapolasi yang dipakai adalah `full - fac`, arah yang sebelumnya menghasilkan skor public terbaik tim (`qc_00_ext1.5_v4_fac_full_only.csv`, public 0.56171). `fac` adalah model yang dilatih tanpa blok putaran kedua (`r2_`, `b2_`, `a2_`, `e2_`), `full` memakai semuanya. Rasio 0.75 sampai 1.5 berada di plateaux yang lebar menurut `agas/v61/training/study_blend_v61.py`, jadi pilihan rasio tidak bergantung pada satu puncak OOF.

In [6]:
if RUN_TRAINING:
    os.system(f'python3 {TRAINING / "study_blend_v61.py"}')
    os.system(f'python3 {TRAINING / "make_submission_v61_final.py"}')

summary = json.load(open(OUT / 'summary_v61_final.json'))
cands = pd.DataFrame([{k: v for k, v in c.items() if k != 'problems'}
                     for c in summary['candidates']])
cands

,name,auc,ks,file,spearman_anchor
0,alt_ext0.5_tm30_lean,0.846699,0.528624,/Users/agas/Documents/ITB/Lomba/dsc-logika-ui-...,0.985917
1,ext0.75_full_fac,0.846780,0.527968,/Users/agas/Documents/ITB/Lomba/dsc-logika-ui-...,0.986582
2,consensus,0.846775,0.527879,/Users/agas/Documents/ITB/Lomba/dsc-logika-ui-...,0.986438
3,ext1.5_full_fac,0.845662,0.527838,/Users/agas/Documents/ITB/Lomba/dsc-logika-ui-...,0.981550
4,ext1.0_full_fac,0.846487,0.527660,/Users/agas/Documents/ITB/Lomba/dsc-logika-ui-...,0.985179
5,meta_logit,0.847157,0.525870,/Users/agas/Documents/ITB/Lomba/dsc-logika-ui-...,0.989369
6,topmix20,0.847177,0.525791,/Users/agas/Documents/ITB/Lomba/dsc-logika-ui-...,0.989121
7,anchor5_consensus,NaN,NaN,/Users/agas/Documents/ITB/Lomba/dsc-logika-ui-...,0.999966
8,anchor15_consensus,NaN,NaN,/Users/agas/Documents/ITB/Lomba/dsc-logika-ui-...,0.999696
9,anchor35_consensus,NaN,NaN,/Users/agas/Documents/ITB/Lomba/dsc-logika-ui-...,0.998340


## 5. Validasi file submission

Setiap kandidat diperiksa ulang di sini: jumlah baris 30000, `application_id` unik dan sama dengan `data/test_applications.csv`, serta `default_probability` berada di rentang [0, 1].

In [7]:
test_apps = pd.read_csv(ROOT / 'data/test_applications.csv', usecols=['application_id'])
ref = set(test_apps.application_id)

rows = []
for path in sorted(OUT.glob('submission_v61_*.csv')):
    d = pd.read_csv(path)
    ok = (len(d) == 30000 and d.application_id.is_unique and set(d.application_id) == ref
          and d.default_probability.between(0, 1).all() and not d.default_probability.isna().any())
    rows.append({'file': path.name, 'rows': len(d), 'ok': ok,
                 'min': round(d.default_probability.min(), 6),
                 'max': round(d.default_probability.max(), 6)})
pd.DataFrame(rows)

,file,rows,ok,min,max
0,submission_v61_alt_ext0.5_tm30_lean.csv,30000,True,0.000033,1.0
1,submission_v61_anchor15_consensus.csv,30000,True,0.000033,1.0
2,submission_v61_anchor15_v61.csv,30000,True,0.000033,1.0
3,submission_v61_anchor30_v61.csv,30000,True,0.000033,1.0
4,submission_v61_anchor35_consensus.csv,30000,True,0.000033,1.0
5,submission_v61_anchor50_v61.csv,30000,True,0.000033,1.0
6,submission_v61_anchor5_consensus.csv,30000,True,0.000033,1.0
7,submission_v61_cat_full.csv,30000,True,0.000033,1.0
8,submission_v61_consensus.csv,30000,True,0.000033,1.0
9,submission_v61_core.csv,30000,True,0.000033,1.0


## 6. Urutan submit

1. `submission_v61_consensus.csv`, rata rank dari empat rasio ekstrapolasi yang bertetangga plus meta logit.
2. `submission_v61_ext1.5_full_fac.csv`, rasio yang sama dengan file public terbaik tim.
3. `submission_v61_ext0.75_full_fac.csv`, konservatif di ujung lain plateaux.
4. `submission_v61_meta_logit.csv`, AUC OOF tertinggi tanpa ekstrapolasi, dipakai sebagai kontrol.
5. `submission_v61_anchor15_consensus.csv`, hanya 15 persen consensus baru di atas anchor terbukti.

Hasil public yang dilaporkan user: `submission_v61_alt_ext0.5_tm30_lean.csv` mendapat **0.56312**, melewati seluruh kandidat lain. File itu kini menjadi jangkar untuk v62 dan disimpan di `agas/v61/output/`.